# AI Company Enrichment Colab

Run the install cell, then run the enrichment cell and paste a JSON array of company URLs when prompted.

In [ ]:
!pip -q install requests beautifulsoup4 google-generativeai

In [ ]:
import json, os, re, time, requests, certifi, xml.etree.ElementTree as ET
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
try:
    import google.generativeai as genai
except Exception:
    genai = None

SCHEMA_KEYS = ['website_name','company_name','address','mobile_number','mail','core_service','target_customer','probable_pain_point','outreach_opener']
HEADERS = {'User-Agent':'Mozilla/5.0 Chrome/124 Safari/537.36','Accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8','Accept-Language':'en-US,en;q=0.9'}
KEYWORDS = {'contact':['contact','support','reach-us','get-in-touch'], 'about':['about','company','who-we-are','our-story'], 'services':['services','solutions','products','platform','industries','what-we-do']}
PRIORITY = {'home':100,'contact':90,'about':80,'services':70,'other':10}

def empty_profile(url='', website_name=''):
    name = website_name.strip() or name_from_domain(url)
    return {'website_name':name,'company_name':name,'address':'','mobile_number':'','mail':[],'core_service':'','target_customer':'','probable_pain_point':'','outreach_opener':''}

def normalize_url(url):
    url = (url or '').strip()
    if not url: return ''
    if not re.match(r'^https?://', url, re.I) and '.' not in url and '/' not in url:
        compact = re.sub(r'[^A-Za-z0-9]+', '', url)
        url = f'www.{compact}.com'
    if not re.match(r'^https?://', url, re.I): url = 'https://' + url
    return urlparse(url).geturl().rstrip('/')

def name_from_domain(url):
    host = urlparse(normalize_url(url)).netloc.replace('www.','')
    return re.sub(r'[-_]+',' ', host.split('.')[0]).title() if host else ''

def humanize_name(name):
    return ' '.join(word.capitalize() for word in re.sub(r'[-_]+', ' ', name or '').split())

def fetch(url, deadline, timeout=10):
    if time.monotonic() >= deadline: return None
    try:
        remaining = max(1.0, min(timeout, deadline - time.monotonic()))
        r = requests.get(url, headers=HEADERS, timeout=remaining, allow_redirects=True, verify=certifi.where())
        return r if r.status_code < 400 else None
    except Exception:
        return None

def clean_text(html):
    soup = BeautifulSoup(html or '', 'html.parser')
    title = soup.title.get_text(' ', strip=True) if soup.title else ''
    for tag in soup(['script','style','noscript','svg','canvas','iframe']): tag.decompose()
    for selector in ['nav','footer','header',"[role='navigation']",'.cookie','#cookie']:
        for tag in soup.select(selector): tag.decompose()
    lines, seen = [], set()
    junk = re.compile(r'^(accept|reject|privacy policy|terms|cookie|subscribe|sign in|log in)$', re.I)
    for line in soup.get_text('\n', strip=True).splitlines():
        line = re.sub(r'\s+', ' ', line).strip()
        key = line.lower()
        if len(line) < 3 or junk.match(line) or key in seen: continue
        seen.add(key); lines.append(line)
    return '\n'.join(lines[:700]), title

def page_kind(url, is_home=False):
    if is_home: return 'home'
    low = url.lower()
    for kind, terms in KEYWORDS.items():
        if any(term in low for term in terms): return kind
    return 'other'

def sitemap_urls(base, deadline):
    r = fetch(urljoin(base + '/', 'sitemap.xml'), deadline, 6)
    if not r or not r.text: return []
    urls = []
    try:
        root = ET.fromstring(r.text)
        for elem in root.iter():
            if elem.tag.endswith('loc') and elem.text and urlparse(elem.text.strip()).netloc == urlparse(base).netloc:
                urls.append(elem.text.strip())
    except Exception:
        urls = re.findall(r'<loc>(.*?)</loc>', r.text, flags=re.I)
    return urls[:80]

def home_links(home_url, html):
    host = urlparse(home_url).netloc
    soup = BeautifulSoup(html or '', 'html.parser')
    urls = []
    for a in soup.find_all('a', href=True):
        href = a.get('href','')
        if href.startswith(('mailto:','tel:','#','javascript:')): continue
        absolute = urljoin(home_url + '/', href).split('#')[0].rstrip('/')
        if urlparse(absolute).netloc == host: urls.append(absolute)
    return list(dict.fromkeys(urls))[:120]

def link_score(url):
    kind = page_kind(url)
    low = url.lower()
    score = PRIORITY.get(kind, 0)
    for terms in KEYWORDS.values():
        for term in terms:
            if term in low: score += 30
    return score, kind

def choose_urls(base, home_html, deadline, max_pages=6):
    candidates = list(dict.fromkeys([base] + sitemap_urls(base, deadline) + home_links(base, home_html)))
    scored = []
    for u in candidates:
        kind = page_kind(u, u.rstrip('/') == base.rstrip('/'))
        score, _ = link_score(u)
        scored.append((PRIORITY.get(kind,0) + score, kind, u))
    scored.sort(reverse=True)
    selected, kinds = [], set()
    for _, kind, u in scored:
        if kind in {'home','contact','about','services'} and kind not in kinds:
            selected.append((u, kind)); kinds.add(kind)
        if len(selected) >= max_pages: break
    for _, kind, u in scored:
        if len(selected) >= max_pages: break
        if u not in [x for x, _ in selected]: selected.append((u, kind))
    return selected

def candidate_start_urls(url):
    normalized = normalize_url(url)
    parsed = urlparse(normalized)
    host = parsed.netloc
    candidates = [normalized]
    original = (url or '').strip()
    if not re.match(r'^https?://', original, re.I) and '.' not in original and '/' not in original:
        compact = re.sub(r'[^A-Za-z0-9]+', '', original).lower()
        hyphenated = re.sub(r'[^A-Za-z0-9]+', '-', original).strip('-').lower()
        for domain in [compact, hyphenated]:
            if domain:
                candidates.append(f'https://www.{domain}.com')
                candidates.append(f'https://{domain}.com')
    if host.startswith('www.'):
        candidates.append(normalized.replace('://www.', '://', 1))
    elif host:
        candidates.append(normalized.replace('://', '://www.', 1))
    if parsed.scheme == 'https':
        candidates.append(normalized.replace('https://', 'http://', 1))
    return list(dict.fromkeys(candidates))

def scrape_site(url, deadline):
    start_urls = candidate_start_urls(url)
    if not start_urls: return [], ''
    r, normalized = None, start_urls[0]
    for candidate in start_urls:
        r = fetch(candidate, deadline)
        if r:
            normalized = candidate
            break
    if not r: return [], normalized
    final = r.url.rstrip('/')
    text, title = clean_text(r.text)
    pages = [{'url':final,'kind':'home','html':r.text,'text':text,'title':title}]
    for page_url, kind in choose_urls(final, r.text, deadline):
        if page_url.rstrip('/') == final.rstrip('/'): continue
        if time.monotonic() >= deadline: break
        time.sleep(0.25)
        pr = fetch(page_url, deadline, 8)
        if not pr: continue
        ptext, ptitle = clean_text(pr.text)
        if len(ptext) < 80: continue
        pages.append({'url':pr.url.rstrip('/'),'kind':kind,'html':pr.text,'text':ptext,'title':ptitle})
        if len(pages) >= 6: break
    return pages, final

def organization_tokens(final_url='', company_name=''):
    parsed = urlparse(final_url)
    parts = [p for p in parsed.path.split('/') if len(p) >= 3]
    words = re.findall(r'[A-Za-z0-9]+', company_name.lower())
    tokens = [p.lower() for p in parts if p.lower() not in {'www','com','org','net','html'}]
    tokens.extend(w for w in words if len(w) >= 4)
    if len(words) >= 2:
        acronym = ''.join(w[0] for w in words if w)
        if len(acronym) >= 3: tokens.append(acronym)
    return list(dict.fromkeys(tokens))

def extract_emails(text, final_url='', company_name=''):
    emails = re.findall(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', text or '')
    out = []
    for e in emails:
        e = e.strip('.,;:()[]{}').lower()
        if e.endswith(('example.com','domain.com','email.com')): continue
        if e not in out: out.append(e)
    host = urlparse(final_url).netloc.replace('www.', '').lower()
    tokens = organization_tokens(final_url, company_name)
    freemail = {'gmail.com','yahoo.com','hotmail.com','outlook.com','live.com','icloud.com'}
    org_specific = []
    for e in out:
        local, domain = e.rsplit('@', 1)
        if domain in freemail: continue
        if any(re.search(rf'(^|[._-]){re.escape(t)}($|[._-])', f'{local}.{domain}') for t in tokens): org_specific.append(e)
    if org_specific: return org_specific[:5]
    official = []
    for e in out:
        domain = e.rsplit('@', 1)[1]
        if domain in freemail: continue
        if host and (domain == host or domain.endswith('.' + host) or host.endswith('.' + domain)): official.append(e)
    return (official or out)[:5]

def extract_phones(text):
    phones = []
    for raw in re.findall(r'(?:(?:\+|00)\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?){2,5}\d{3,4}', text or ''):
        digits = re.sub(r'\D','',raw)
        if 8 <= len(digits) <= 15 and raw.strip() not in phones: phones.append(raw.strip())
    return phones[:3]

def extract_address(text):
    words = re.compile(r'\b(street|st\.|road|rd\.|avenue|ave\.|drive|dr\.|lane|ln\.|suite|floor|building|city|state|zip|postal|india|usa|united states|canada|australia)\b', re.I)
    phone_or_support = re.compile(r'\b(toll free|phone|fax|call|tel|mobile|email|@)\b', re.I)
    lines = [x.strip() for x in (text or '').splitlines() if x.strip()]
    for i in range(len(lines)):
        if re.search(r'\b(address|head office|registered office|corporate office)\b', lines[i], re.I):
            parts = []
            for candidate in lines[i:i+6]:
                digits = len(re.findall(r'\d', candidate))
                if phone_or_support.search(candidate) and digits >= 7: break
                if re.search(r'\b(email|phone|mobile|call|fax)\b', candidate, re.I): break
                parts.append(candidate)
            address = ' '.join(parts).strip(' :-,')
            address = re.sub(r'^(?:\+?\d[\d\s().-]{7,}\d\s*)+', '', address).strip(' :-,')
            if len(address) > 20 and re.search(r'\d', address): return address[:300]
        window = ' '.join(lines[i:i+3])
        digits = len(re.findall(r'\d', window))
        if phone_or_support.search(window) and digits >= 7: continue
        if len(re.findall(r'\d[\d\s().+-]{6,}\d', window)) >= 2: continue
        if len(window) > 35 and words.search(window) and re.search(r'\d', window):
            window = re.sub(r'^(?:\+?\d[\d\s().-]{7,}\d\s*)+', '', window).strip(' :-,')
            return window[:250]
    return ''

def infer_names(pages, final_url):
    candidates = []
    domain_name = name_from_domain(final_url)
    for p in pages:
        if p.get('title'):
            parts = [part.strip() for part in re.split(r'\s*(?:\||-|--|\u2013|\u2014|\u2022)\s*', p['title']) if part.strip()]
            for part in parts:
                score = 60 if domain_name and part.lower() == domain_name.lower() else 25
                candidates.append((score, part))
        soup = BeautifulSoup(p.get('html',''), 'html.parser')
        for selector in ["meta[property='og:site_name']", "meta[name='application-name']"]:
            tag = soup.select_one(selector)
            if tag and tag.get('content'): candidates.append((80, tag['content'].strip()))
        h1 = soup.find('h1')
        if h1: candidates.append((10, h1.get_text(' ', strip=True)))
    candidates.append((50, domain_name))
    marketing_words = re.compile(r'\b(train|deploy|leading|industry|solutions|services|platform|software|data|ai|generative|traditional|welcome|home|contact|about)\b', re.I)
    for _, c in sorted(candidates, key=lambda item: item[0], reverse=True):
        c = re.sub(r'\s+', ' ', c).strip()
        if 2 < len(c) <= 40 and not marketing_words.search(c): return c, c
    return domain_name, domain_name

def compact_context(pages, max_chars=14000):
    order = {'contact':0,'about':1,'services':2,'home':3,'other':4}
    chunks = []
    for p in sorted(pages, key=lambda x: order.get(x.get('kind'), 9)):
        chunks.append(f"URL: {p['url']}\nPAGE_TYPE: {p['kind']}\n{p['text'][:4500]}")
    return '\n\n---\n\n'.join(chunks)[:max_chars]

def local_business_insights(text, company_name):
    lowered = f'{company_name}\n{text}'.lower()
    categories = [
        {'priority':90, 'patterns':[r'\buniversity\b',r'\binstitute\b',r'\bcollege\b',r'\bschool\b',r'\bcampus\b',r'\badmission(?:s)?\b',r'\bprogramme?s?\b',r'\bcourses?\b',r'\bdegree\b',r'\bfaculty\b',r'\bstudents?\b',r'\bplacements?\b',r'\bengineering\b',r'\bmanagement\b',r'\bb\.?\s*tech\b',r'\bm\.?\s*tech\b',r'\bph\.?\s*d\b'], 'service':'Undergraduate and postgraduate education in engineering, technology, management, and career-oriented programs.', 'customer':'Prospective students and parents evaluating technical and professional programs based on admissions, campus location, and placements.', 'pain':'Finding credible technical programs while comparing admissions, placement outcomes, campus quality, and career fit.', 'focus':'programs, admissions, campus location, and placement outcomes', 'tail':'connecting with prospective students who are actively comparing programs and admissions options'},
        {'priority':80, 'patterns':[r'\bpackers?\b',r'\bmovers?\b',r'\bpacking\b',r'\bshifting\b',r'\brelocation\b',r'\btransport(?:ation)?\b',r'\blogistics\b',r'\bfreight\b',r'\bwarehouse\b',r'\bloading\b',r'\bunloading\b',r'\bhousehold goods\b',r'\boffice shifting\b'], 'service':'Packing, moving, relocation, transportation, and logistics support.', 'customer':'Households, offices, and businesses planning local or intercity relocation.', 'pain':'Moving goods safely and on time without damage, delays, or coordination stress.', 'focus':'packing, moving, and relocation services', 'tail':'reaching customers who are actively looking for this support'},
        {'priority':40, 'patterns':[r'\bsoftware\b',r'\bsaas\b',r'\bcloud\b',r'\bapi\b',r'\bplatform\b',r'\bapp(?:lication)?s?\b',r'\bdevops\b'], 'service':'Software, cloud, or platform-based technology services.', 'customer':'Businesses looking to modernize software workflows or digital operations.', 'pain':'Reducing manual work, integration friction, and operational bottlenecks across tools.', 'focus':'software and digital operations'},
        {'priority':30, 'patterns':[r'\bartificial intelligence\b',r'\bgenerative ai\b',r'\bmachine learning\b',r'\bdata annotation\b',r'\bdata analytics\b',r'\bdata science\b',r'\bai\b',r'\bml\b'], 'service':'AI, machine learning, data, or analytics services.', 'customer':'Enterprises and product teams using data-heavy workflows or AI systems.', 'pain':'Turning complex data workflows into accurate, scalable business outcomes.', 'focus':'AI and data services'},
        {'priority':70, 'patterns':[r'\bhealthcare\b',r'\bhospital\b',r'\bclinic\b',r'\bpatient\b',r'\bmedical\b'], 'service':'Healthcare or medical services.', 'customer':'Patients, healthcare providers, or organizations needing medical support.', 'pain':'Improving access, coordination, and reliability across healthcare operations.', 'focus':'healthcare services'},
        {'priority':60, 'patterns':[r'\bfinance\b',r'\bfinancial\b',r'\bloan\b',r'\binsurance\b',r'\bpayment\b',r'\btax\b'], 'service':'Financial, insurance, payment, or advisory services.', 'customer':'Individuals or businesses managing financial decisions and transactions.', 'pain':'Reducing risk, delays, and complexity in financial operations.', 'focus':'financial services'},
    ]
    count, best = max(((sum(1 for p in c['patterns'] if re.search(p, lowered)), c) for c in categories), key=lambda x: (x[0] * 10 + int(x[1]['priority']), x[0]))
    if not count: return {'core_service':'','target_customer':'','probable_pain_point':'','outreach_opener':''}
    tail = best.get('tail', 'reaching people actively looking for this support')
    return {'core_service':best['service'], 'target_customer':best['customer'], 'probable_pain_point':best['pain'], 'outreach_opener':f"Hi {company_name} team, I noticed your website highlights {best['focus']}. I would love to share a quick idea for {tail}."}

def parse_json(raw):
    if not raw: return {}
    s = re.sub(r'^```(?:json)?', '', raw.strip(), flags=re.I).strip()
    s = re.sub(r'```$', '', s).strip()
    try: return json.loads(s)
    except Exception:
        m = re.search(r'\{.*\}', s, flags=re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: return {}
    return {}

def call_gemini(context, facts, deadline):
    key = os.getenv('GEMINI_API_KEY', '').strip()
    if not key or genai is None or time.monotonic() >= deadline: return {}
    genai.configure(api_key=key)
    model = genai.GenerativeModel('gemini-1.5-flash')
    prompt = f'''Return ONLY valid JSON with exactly these keys: {SCHEMA_KEYS}.
Do not invent emails, phones, or addresses. Use factual extraction for factual fields. Missing facts become empty strings or []. Business insights must come only from website text. First identify organization type. If the site is a college, university, institute, school, admissions page, or campus page, classify it as education even if it mentions AI, data science, engineering, or software courses. Do not classify a business as AI/data just because AI appears as a course, department, blog topic, client use case, or incidental phrase. core_service must name actual programs, departments, services, specializations, industries, or products visible on the site. target_customer must be specific about geography, entry path, audience segment, buyer type, or use case. probable_pain_point must be specific to this organization niche, geography, positioning, or offering. outreach_opener must reference one concrete detail from the website and must not use vague phrases like I would love to share an idea. Outreach opener must not use fake metrics.
Factual extraction: {json.dumps(facts, ensure_ascii=False)}
Cleaned website text: {context}'''
    try:
        remaining = max(2, min(8, int(deadline - time.monotonic())))
        resp = model.generate_content(prompt, generation_config={'temperature':0.2,'response_mime_type':'application/json'}, request_options={'timeout':remaining})
        return parse_json(getattr(resp, 'text', ''))
    except Exception:
        return {}

def stable(data):
    out = {k:data.get(k, [] if k == 'mail' else '') for k in SCHEMA_KEYS}
    if not isinstance(out['mail'], list): out['mail'] = [str(out['mail'])] if out['mail'] else []
    out['mail'] = [str(x).strip() for x in out['mail'] if str(x).strip()]
    for k in SCHEMA_KEYS:
        if k != 'mail': out[k] = '' if out[k] is None else str(out[k]).strip()
    return out

def enrich_company(url, website_name='', total_timeout=20):
    deadline = time.monotonic() + total_timeout
    profile = empty_profile(normalize_url(url), website_name)
    try:
        pages, final_url = scrape_site(url, deadline)
        if not pages: return stable(profile)
        all_text = '\n'.join(p['text'] for p in pages)
        contact_text = '\n'.join(p['text'] for p in pages if p['kind'] == 'contact') or all_text
        phones = extract_phones(contact_text)
        address = extract_address(contact_text)
        inferred_site, inferred_company = infer_names(pages, final_url)
        display_company = humanize_name(website_name) if website_name.strip() and ' ' in website_name.strip() and ' ' not in inferred_company else inferred_company
        emails = extract_emails(all_text, final_url, display_company)
        facts = {'website_name': website_name.strip() or inferred_site, 'company_name': display_company, 'address': address, 'mobile_number': phones[0] if phones else '', 'mail': emails, 'source_url': final_url}
        profile.update(facts)
        ai = call_gemini(compact_context(pages), facts, deadline)
        fallback = local_business_insights(all_text, display_company)
        merged = {**profile, **ai}
        for key, value in fallback.items():
            if value: merged[key] = value
        merged['mail'] = emails; merged['mobile_number'] = phones[0] if phones else ''; merged['address'] = address
        merged['website_name'] = website_name.strip() or merged.get('website_name') or inferred_site
        merged['company_name'] = merged.get('company_name') or display_company
        return stable(merged)
    except Exception:
        return stable(profile)

raw = input('Paste JSON array of company URLs: ')
urls = json.loads(raw)
results = [enrich_company(url) for url in urls]
print(json.dumps(results, indent=2, ensure_ascii=False))
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
